In [ ]:
!pip install pyrudof

In [ ]:
# Leemos unas pocas tripletas desde un objeto str

from pyrudof import Rudof, RudofConfig, RDFFormat, ReaderMode

rudof = Rudof(RudofConfig())

rudof.read_data_str(input="""
    prefix : <http://example.org/>
    :ok :name "alice" .
    :ko :name "ecila" .
""",
                    format=RDFFormat.Turtle,
                    base=None,
                    reader_mode=ReaderMode.Lax,
                    merge=False)


# Las serializamos en N-triples
print(rudof.serialize_data(RDFFormat.Turtle))

In [ ]:
#Rudof mantendrá el grafo en memoria que le hayamso cargado. Podemos añadir nuevas tripletas a este grafo a través de
# leer nuevo contenido con el flag de "merge" a True

rudof.read_data_str(input="""
    prefix : <http://example.org/>
    :ko :name "other" .
""",
                    format=RDFFormat.Turtle,
                    base=None,
                    reader_mode=ReaderMode.Lax,
                    merge=True)
# Ten en cuenta que las nuevas tripletas se añaden como un nuevo grafo independiente, no como contexto del grafo anterior.
# Si vas a usar los mismos prefijos, tienes que volver a declararlos. De lo contrario se considerará sintaxis inválida.


print(rudof.serialize_data(RDFFormat.NTriples))

In [ ]:
# Podemos cargar y ejecutar una consulta SPARQL contra los datos que Rudof tenga en memoria

result = rudof.run_query_str("""
select * where {
  ?s ?p ?o .
}
""")
# Mostrando resultado de forma tabular legible para humanos
print(result.show())
print("-----")
# Mostrando bindings de variables y valores como un objeto JSON fácil de parsear
print(result.as_json())


In [53]:
# Podemos también operar contra datos en archivos locales
# (para lo cual primero vamos a cloner el repo de la asignatura para usar ejemplos disponibles en el mismo de forma local)

In [ ]:
!git clone https://github.com/cursosLabra/miw_websem.git

In [ ]:
base_path = "/content/miw_websem/Introduction/"
rudof.read_data(base_path + "example1.ttl",
                format=RDFFormat.Turtle,
                base=None,
                reader_mode=ReaderMode.Lax,
                merge=False)  # Merge a False, para reemplazar los datos antiguos

# Muestra de los datos leidos en Ntriples:
print(rudof.serialize_data(RDFFormat.NTriples))
print("-------")
# Ejecución de la misma consulta sobre los datos leídos:
result = rudof.run_query_str("""
select * where {
  ?s ?p ?o .
}
""")

print(result.show())


In [ ]:
# Podemos también hacer consultas a un endpoint:

rudof.use_endpoint("https://query.wikidata.org/sparql")

# Al indicar que usaremos un endpoint, rudof "olvida" los datos que teníamos parseados.
# Las operaciones de consulta se ejecutarán sobre estos nuevos datos.

result = rudof.run_query_str("""

PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

select ?s where {
  ?s wdt:P31 wd:Q6256 .   # P31 --> instance of. Q6256 ---> country
} LIMIT 10
""")
print(result.show())



In [ ]:
# Si queremos ejecutar consultas CONSTRUCT en lugar de SELECT, podemos hacerlo.
# Pero usando un método distinto.

from pyrudof import QueryResultFormat

result = rudof.run_query_construct_str("""
    PREFIX wd:  <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX :    <http://example.org/>

    CONSTRUCT {
        ?p a :Person ;
           :name ?person .
    } WHERE {
        ?p wdt:P31 wd:Q5 ;
           rdfs:label ?person .
        FILTER (lang(?person) = "en")
    }
    LIMIT 5
""", QueryResultFormat.Turtle)

print(result)

In [ ]:
# Rudof ofece funciones más allá del mero parseo, serialización y consulta de RDF: generación de datos sintéticos, validación ShEx, SHACL,
# generación de visualizaciones tras validación... pero veremos ejemplos de eso en sesiones futuras.